# 01 — MHA-13L baseline training run

This notebook benchmarks a 13-layer model with standard Multi-Head Attention (8 query heads and 8 K/V heads), RoPE, the 16,384-token SentencePiece vocabulary, and tied input/output embeddings. It is close to the 50M parameter limit. Run it before the GQA notebook so all experiments can be compared with the same seed, batch size, sequence length, and number of optimization steps.

The run uses synthetic token batches to measure architecture and kernel performance without introducing dataset or network variability. It is a systems smoke test, not a quality evaluation. Metrics are logged to Weights & Biases and also saved locally as JSON.

In [ ]:
from copy import deepcopy
import json
from pathlib import Path
import time

import torch
import torch.nn.functional as F
import wandb

from llm_mini_lab.models.gpt import GPTModel
from llm_mini_lab.training.core import GPT_CONFIG_50M

SEED = 42
BATCH_SIZE = 2
SEQUENCE_LENGTH = 128
WARMUP_STEPS = 2
TRAIN_STEPS = 10
LEARNING_RATE = 3e-4
WANDB_PROJECT = "50M-LLM-GQA"
WANDB_GROUP = "mha-vs-gqa-16k-rope"

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
amp_dtype = torch.bfloat16 if device.type == "cuda" and torch.cuda.is_bf16_supported() else torch.float16
print("Device:", device)
if device.type != "cuda":
    print("CUDA is not available: SDPA will run, but this run cannot verify a CUDA fused kernel or peak VRAM.")
wandb.login()


In [ ]:
def count_parameters(model):
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)

def run_training_trial(name, config):
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    model = GPTModel(config).to(device).train()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    run = wandb.init(
        project=WANDB_PROJECT,
        group=WANDB_GROUP,
        name=name,
        config={**config, "batch_size": BATCH_SIZE, "sequence_length": SEQUENCE_LENGTH, "warmup_steps": WARMUP_STEPS, "train_steps": TRAIN_STEPS, "learning_rate": LEARNING_RATE},
    )
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda" and amp_dtype == torch.float16)
    generator = torch.Generator(device=device).manual_seed(SEED)
    tokens = torch.randint(0, config["vocab_size"], (BATCH_SIZE, SEQUENCE_LENGTH + 1), device=device, generator=generator)

    def step():
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=device.type == "cuda"):
            logits = model(tokens[:, :-1])
            loss = F.cross_entropy(logits.flatten(0, 1).float(), tokens[:, 1:].flatten())
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        return loss.detach().item()

    for _ in range(WARMUP_STEPS):
        step()
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()

    losses = []
    start = time.perf_counter()
    for step_index in range(TRAIN_STEPS):
        loss = step()
        losses.append(loss)
        run.log({"train/loss": loss}, step=step_index)
    if device.type == "cuda":
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - start

    result = {
        "name": name,
        "device": torch.cuda.get_device_name(0) if device.type == "cuda" else str(device),
        "parameters": count_parameters(model),
        "layers": config["n_layers"],
        "query_heads": config["n_heads"],
        "kv_heads": config["n_kv_heads"],
        "batch_size": BATCH_SIZE,
        "sequence_length": SEQUENCE_LENGTH,
        "steps": TRAIN_STEPS,
        "initial_loss": losses[0],
        "final_loss": losses[-1],
        "milliseconds_per_step": 1000 * elapsed / TRAIN_STEPS,
        "tokens_per_second": BATCH_SIZE * SEQUENCE_LENGTH * TRAIN_STEPS / elapsed,
        "peak_allocated_gb": torch.cuda.max_memory_allocated() / 1e9 if device.type == "cuda" else None,
    }
    run.summary.update(result)
    run.finish()
    return result


In [ ]:
mha_config = deepcopy(GPT_CONFIG_50M)
mha_config.update({"vocab_size": 16_384, "tokenizer_name": "sp16384", "positional_encoding": "rope", "n_layers": 13})
mha_config["n_kv_heads"] = mha_config["n_heads"]
mha_result = run_training_trial("MHA-13L", mha_config)
mha_result


In [ ]:
results_dir = Path("results")
results_dir.mkdir(exist_ok=True)
result_path = results_dir / "mha_baseline.json"
result_path.write_text(json.dumps(mha_result, indent=2), encoding="utf-8")
print("Saved:", result_path.resolve())
